# Data Prep v3 — 端到端数据准备管线（方案三：CWE-MIXED 单桶 + 反向索引 τ 阈值）

**方案三**是**方案四**的改良版，两者唯一区别在于 **代理查询集不再按 CWE 分类**：

1. 语料 & 查询集构建（Vul + BFP + white/gray/black query_set）—— **与 v4 一致**
2. 三个向量库构建（BFP-Proxy / BFP-APR / Vul-Vinj）—— **与 v4 一致**
3. 代理查询集 & 测试查询集（BFP↔CVE 匹配 + **CWE-MIXED 单桶汇总去重**（取自 v2）+ 8:2 划分）
4. 投毒目标 & 正样本选择（**反向索引 + τ 阈值过滤**（取自 v4），但作用于单一 `CWE-MIXED` 桶）

> 与 v4 的关键差异：Section 3.2 改用 v2 的单桶 `CWE-MIXED` 汇总逻辑；Section 4 沿用 v4 的反向索引 + τ 阈值方法（不引入 v2 的聚类/预算/LLM-Reranker），但遍历对象由逐 CWE 改为单一 `CWE-MIXED`。

> **使用方法**：按顺序运行所有 cell；每个写盘/建库步骤均有幂等检查，已完成则自动跳过。

## Section 0 — 全局初始化

In [1]:
# ========== Section 0: 全局初始化 ==========
import os
import sys
import gc
import json
import random
import torch
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# ── 项目根目录上溯 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

from src.utils.io import load_json, load_jsonl, load_json_stream, save_json, load_yaml
from src.utils.log import setup_logging
from src.models.retriever.Base import create_embedder
from src.rag.bfp.milvus_client import BFPMilvusClient
from src.rag.bfp.retriever import BFPRetriever
from src.rag.vul.milvus_client import VulMilvusClient
from src.rag.vul.retriever import VulRetriever

setup_logging()


# ── 嵌入模型按需加载/释放的辅助函数 ──
def load_embedder_from_db_config(db_config: dict):
    """从数据库配置的 database.model 字段加载嵌入模型。"""
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
    cfg_path = PROJECT_ROOT / db_config["database"]["model"]
    return create_embedder(load_yaml(str(cfg_path)))

def release_embedder(emb) -> None:
    """释放嵌入模型并清空 GPU 显存。"""
    del emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("[GPU] 嵌入模型已释放")

PROJECT_ROOT: <PROJECT_ROOT>


## Section 1 — 语料 & 查询集构建

### Section 1.1 — Vul 语料构建

In [2]:
# ========== Section 1.1: Vul 语料构建 ==========
# 产出：data/corpus/vul/clustered_rag_data.jsonl

VUL_CORPUS_OUT = PROJECT_ROOT / "data/corpus/vul/clustered_rag_data.jsonl"

if VUL_CORPUS_OUT.exists():
    print(f"[SKIP] Vul 语料已存在：{VUL_CORPUS_OUT}")
    df_vul = pd.read_json(str(VUL_CORPUS_OUT), lines=True)
    print(f"       加载 {len(df_vul)} 条")
else:
    # ── 1. 加载原始数据集（已由 vul/dataset_formatting.ipynb 预先格式化）──
    bigvul_df   = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/BigVul.jsonl"),   lines=True)
    cvefixes_df = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/CVEfixes.jsonl"), lines=True)
    reposvul_df = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/ReposVul_v2.jsonl"), lines=True)
    df_vul = pd.concat([bigvul_df, cvefixes_df, reposvul_df], ignore_index=True)

    print(f"原始合并: {len(df_vul)} 条")

    # ── 2. 按 vul_code + cwe_id 去重（同一漏洞代码可能对应不同 CWE，需保留）──
    before = len(df_vul)
    df_vul = df_vul.drop_duplicates(subset=["vul_code", "cwe_id"], keep="first").reset_index(drop=True)
    print(f"去重: {before} → {len(df_vul)} (移除 {before - len(df_vul)})")

    # ── 3. 嵌入 + KMeans 聚类 ──
    all_codes = df_vul["vul_code"].astype(str).tolist()
    vul_embedder = load_embedder_from_db_config(
        {"database": {"model": "configs/models/harrier-oss-v1-0.6b.yml"}}
    )
    encoded = np.asarray(vul_embedder.embed_documents(all_codes))
    release_embedder(vul_embedder)  # KMeans 不需要 GPU，提前释放显存
    print(f"嵌入维度: {encoded.shape}")

    kmeans = KMeans(n_clusters=5, random_state=42)
    kmeans.fit(encoded)
    df_vul["cluster_id"] = kmeans.labels_
    print(f"KMeans 聚类完成，各簇分布:\n{df_vul['cluster_id'].value_counts().sort_index()}")

    # ── 4. 写出 ──
    VUL_CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)
    df_vul.to_json(str(VUL_CORPUS_OUT), orient="records", lines=True)
    print(f"✅ 已写入 {len(df_vul)} 条 → {VUL_CORPUS_OUT}")

df_vul.head()

[SKIP] Vul 语料已存在：<PROJECT_ROOT>/data/corpus/vul/clustered_rag_data.jsonl
       加载 17576 条


,id,source,language,cwe_id,cve_id,cwe_desc,cve_desc,vul_code,fixed_code,diff,cluster_id
0,0,BigVul,C,CWE-264,CVE-2015-8467,"- CWE ID: CWE-264\n- Name: Permissions, Privil...",The samldb_check_user_account_control_acl func...,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...,"--- vul_code\n+++ fixed_code\n@@ -9,12 +9,15 @...",2
1,1,BigVul,C,CWE-119,CVE-2015-8382,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The match function in pcre_exec.c in PCRE befo...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,"--- vul_code\n+++ fixed_code\n@@ -87,6 +87,7 @...",3
2,2,BigVul,C,CWE-119,CVE-2013-6712,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The scan function in ext/date/lib/parse_iso_in...,static int scan(Scanner *s) {\n uchar *cursor...,static int scan(Scanner *s) {\n uchar *cursor...,"--- vul_code\n+++ fixed_code\n@@ -147,7 +147,7...",3
3,3,BigVul,C,CWE-310,CVE-2013-6449,- CWE ID: CWE-310\n- Name: Cryptographic Issue...,The ssl_get_algorithm2 function in ssl/s3_lib....,long ssl_get_algorithm2(SSL *s) {\n long alg2...,long ssl_get_algorithm2(SSL *s) {\n long alg2...,"--- vul_code\n+++ fixed_code\n@@ -1,6 +1,6 @@\...",4
4,4,BigVul,C,CWE-119,CVE-2013-6420,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The asn1_time_to_time_t function in ext/openss...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,"--- vul_code\n+++ fixed_code\n@@ -13,20 +13,30...",3


### Section 1.2 — BFP 语料构建

In [3]:
# ========== Section 1.2: BFP 语料构建 ==========
# 产出：data/corpus/bfp/rag_data.jsonl

CORPUS_SIZE = 50000
SEED = 42
BFP_CORPUS_OUT = PROJECT_ROOT / "data/corpus/bfp/rag_data.jsonl"

# ── 工具函数 ──
def extract_fields(record: dict, source: str) -> dict:
    return {
        "language": record["language"],
        "buggy_code": record["buggy_code"],
        "fixed_code": record["fixed_code"],
        "source": source,
    }

def assign_ids(data: list) -> list:
    return [{"id": i} | item for i, item in enumerate(data, 1)]

def write_jsonl(path, data: list):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"  ✅ 已写入 {len(data)} 条 → {path}")

if BFP_CORPUS_OUT.exists():
    print(f"[SKIP] BFP 语料已存在：{BFP_CORPUS_OUT}")
    corpus_dataset = load_jsonl(str(BFP_CORPUS_OUT))
    print(f"       加载 {len(corpus_dataset)} 条")
else:
    random.seed(SEED)

    # ── 读取原始数据（已由 bfp/dataset_formatting.ipynb 预先格式化）──
    coconut_data = []
    with open(PROJECT_ROOT / "data/raw/bfp/CoCoNut.jsonl", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                coconut_data.append(json.loads(line))
    with open(PROJECT_ROOT / "data/raw/bfp/Codeflaws.json", encoding="utf-8") as f:
        codeflaws_data = json.load(f)
    with open(PROJECT_ROOT / "data/raw/bfp/Deepfix.json", encoding="utf-8") as f:
        deepfix_data = json.load(f)

    random.shuffle(coconut_data)
    random.shuffle(codeflaws_data)
    random.shuffle(deepfix_data)

    print(f"原始数据: CoCoNut={len(coconut_data)}, Codeflaws={len(codeflaws_data)}, DeepFix={len(deepfix_data)}")

    # ── 按占比分配语料库样本数 ──
    TOTAL_RAW = len(coconut_data) + len(codeflaws_data) + len(deepfix_data)
    CORPUS_COCONUT   = round(CORPUS_SIZE * len(coconut_data)   / TOTAL_RAW)
    CORPUS_CODEFLAWS = round(CORPUS_SIZE * len(codeflaws_data) / TOTAL_RAW)
    CORPUS_DEEPFIX   = CORPUS_SIZE - CORPUS_COCONUT - CORPUS_CODEFLAWS

    print(f"语料库分配: CoCoNut={CORPUS_COCONUT}, Codeflaws={CORPUS_CODEFLAWS}, DeepFix={CORPUS_DEEPFIX}")

    corpus_coconut   = coconut_data[:CORPUS_COCONUT]
    corpus_codeflaws = codeflaws_data[:CORPUS_CODEFLAWS]
    corpus_deepfix   = deepfix_data[:CORPUS_DEEPFIX]

    corpus_dataset = assign_ids(
        [extract_fields(r, "CoCoNut")   for r in corpus_coconut] +
        [extract_fields(r, "Codeflaws") for r in corpus_codeflaws] +
        [extract_fields(r, "DeepFix")   for r in corpus_deepfix]
    )
    assert len(corpus_dataset) == CORPUS_SIZE

    write_jsonl(BFP_CORPUS_OUT, corpus_dataset)

print(f"BFP 语料库规模: {len(corpus_dataset)} 条")

[SKIP] BFP 语料已存在：<PROJECT_ROOT>/data/corpus/bfp/rag_data.jsonl


2026-06-03 23:15:33 [INFO] src.utils.io: Loaded 50000 records from <PROJECT_ROOT>/data/corpus/bfp/rag_data.jsonl


       加载 50000 条
BFP 语料库规模: 50000 条


### Section 1.3 — BFP 查询集构建（white / gray / black）

In [4]:
# ========== Section 1.3: BFP 查询集构建 ==========
# 产出：
#   data/query/white/query_set.jsonl  （完全 ⊆ 语料库）
#   data/query/gray/query_set.jsonl   （OVERLAP_RATE 来自语料库 + 余下来自语料库外）
#   data/query/black/query_set.jsonl  （全部来自语料库外）

# QUERY_SIZE    = 12500
QUERY_SIZE    = 25000
OVERLAP_RATE  = 0.1
SEED          = 42

WHITE_QUERY_OUT = PROJECT_ROOT / "data/query/white/query_set.jsonl"
GRAY_QUERY_OUT  = PROJECT_ROOT / "data/query/gray/query_set.jsonl"
BLACK_QUERY_OUT = PROJECT_ROOT / "data/query/black/query_set.jsonl"

all_done = WHITE_QUERY_OUT.exists() and GRAY_QUERY_OUT.exists() and BLACK_QUERY_OUT.exists()

if all_done:
    print("[SKIP] 三套查询集已存在")
    white_dataset = load_jsonl(str(WHITE_QUERY_OUT))
    gray_dataset  = load_jsonl(str(GRAY_QUERY_OUT))
    black_dataset = load_jsonl(str(BLACK_QUERY_OUT))
    print(f"       white={len(white_dataset)}, gray={len(gray_dataset)}, black={len(black_dataset)}")
else:
    rng = random.Random(SEED)

    # ── corpus_pool：直接使用已生成的语料库 ──
    corpus_pool = corpus_dataset[:]  # 来自 Section 1.2，已格式化

    # ── outside_pool：原始数据集 − 语料库 ──
    # 用 buggy_code 做精确去重（语料库内条目均来自原始数据集前 N 条）
    corpus_buggy_set = {item["buggy_code"] for item in corpus_pool}

    # 重新读取原始数据（如果全局变量还在则直接用）
    def _load_raw_bfp():
        coc = []
        with open(PROJECT_ROOT / "data/raw/bfp/CoCoNut.jsonl", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    coc.append(json.loads(line))
        with open(PROJECT_ROOT / "data/raw/bfp/Codeflaws.json", encoding="utf-8") as f:
            clf = json.load(f)
        with open(PROJECT_ROOT / "data/raw/bfp/Deepfix.json", encoding="utf-8") as f:
            dpf = json.load(f)
        return coc, clf, dpf

    def infer_source(record: dict) -> str:
        if "user_id" in record:
            return "DeepFix"
        if "bug_type" in record:
            return "Codeflaws"
        return "CoCoNut"

    try:
        # 如果 Section 1.2 的变量还在内存中直接利用
        _raw_all = (
            [extract_fields(r, "CoCoNut")   for r in coconut_data] +
            [extract_fields(r, "Codeflaws") for r in codeflaws_data] +
            [extract_fields(r, "DeepFix")   for r in deepfix_data]
        )
    except NameError:
        coc, clf, dpf = _load_raw_bfp()
        _raw_all = (
            [extract_fields(r, "CoCoNut")   for r in coc] +
            [extract_fields(r, "Codeflaws") for r in clf] +
            [extract_fields(r, "DeepFix")   for r in dpf]
        )

    outside_pool = [
        item for item in _raw_all
        if item["buggy_code"] not in corpus_buggy_set
    ]
    print(f"outside_pool 大小: {len(outside_pool)} 条（已由 bfp/dataset_formatting.ipynb 预先格式化）")

    # ── 抽取三套查询集 ──
    overlap_n = int(QUERY_SIZE * OVERLAP_RATE)   # gray 中来自语料库的部分
    outside_n = QUERY_SIZE - overlap_n           # gray 中来自语料库外的部分

    assert len(corpus_pool) >= QUERY_SIZE, "corpus_pool 不足以支撑 white 查询集"
    assert len(outside_pool) >= QUERY_SIZE + overlap_n, "outside_pool 数量不足"

    white_raw  = rng.sample(corpus_pool, QUERY_SIZE)
    gray_corp  = rng.sample(corpus_pool, overlap_n)
    gray_out   = rng.sample(outside_pool, outside_n)
    gray_raw   = gray_corp + gray_out
    black_raw  = rng.sample(outside_pool, QUERY_SIZE)

    white_dataset = assign_ids(white_raw)
    gray_dataset  = assign_ids(gray_raw)
    black_dataset = assign_ids(black_raw)

    write_jsonl(WHITE_QUERY_OUT, white_dataset)
    write_jsonl(GRAY_QUERY_OUT,  gray_dataset)
    write_jsonl(BLACK_QUERY_OUT, black_dataset)

print(f"\n查询集规模汇总:")
print(f"  white : {len(white_dataset)} 条 → {WHITE_QUERY_OUT}")
print(f"  gray  : {len(gray_dataset)} 条 → {GRAY_QUERY_OUT}")
print(f"  black : {len(black_dataset)} 条 → {BLACK_QUERY_OUT}")

outside_pool 大小: 103367 条（已由 bfp/dataset_formatting.ipynb 预先格式化）
  ✅ 已写入 25000 条 → <PROJECT_ROOT>/data/query/white/query_set.jsonl
  ✅ 已写入 25000 条 → <PROJECT_ROOT>/data/query/gray/query_set.jsonl
  ✅ 已写入 25000 条 → <PROJECT_ROOT>/data/query/black/query_set.jsonl

查询集规模汇总:
  white : 25000 条 → <PROJECT_ROOT>/data/query/white/query_set.jsonl
  gray  : 25000 条 → <PROJECT_ROOT>/data/query/gray/query_set.jsonl
  black : 25000 条 → <PROJECT_ROOT>/data/query/black/query_set.jsonl


## Section 2 — 向量库构建

### Section 2.1 — BFP-Proxy 向量库

配置文件：`configs/database/bfp/proxy.yml`  
语料库和模型文件需要在配置文件中进行设置

In [5]:
# ========== Section 2.1: 构建 BFP-Proxy 向量库 ==========
proxy_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/proxy.yml"))

# 路径绝对化
if not os.path.isabs(proxy_db_config["database"]["uri"]):
    proxy_db_config["database"]["uri"] = str(PROJECT_ROOT / proxy_db_config["database"]["uri"])
if not os.path.isabs(proxy_db_config["data"]["corpus"]):
    proxy_db_config["data"]["corpus"] = str(PROJECT_ROOT / proxy_db_config["data"]["corpus"])
if not os.path.isabs(proxy_db_config["data"]["query"]):
    proxy_db_config["data"]["query"] = str(PROJECT_ROOT / proxy_db_config["data"]["query"])

proxy_milvus = BFPMilvusClient(proxy_db_config)
col_name = proxy_db_config["database"]["collection_name"]

already_built = (
    proxy_milvus.client.has_collection(col_name)
    and proxy_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = proxy_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] BFP-Proxy 向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(proxy_db_config["data"]["corpus"])
    print(f"构建 BFP-Proxy 向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(proxy_db_config)
    proxy_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ BFP-Proxy 向量库构建完成")

proxy_milvus.close()
print("BFP-Proxy 向量库 Done")

2026-06-03 23:17:59 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-06-03 23:17:59 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/proxy/gray/harrier-oss-v1-0.6b.db
2026-06-03 23:17:59 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-06-03 23:18:03 [INFO] src.utils.io: Loaded 25000 records from <PROJECT_ROOT>/data/query/gray/query_set.jsonl


构建 BFP-Proxy 向量库，数据量: 25000 条...


2026-06-03 23:18:03 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-06-03 23:18:03 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-06-03 23:18:03 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-06-03 23:18:03 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-06-03 23:18:03 [INFO] src.utils.device: 预筛选后的设备池: ['cuda:0']
2026-06-03 23:18:04 [INFO] src.utils.device: 设备池探测成功: cuda:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8187 MiB)
<CONDA_ENV>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-03 23:18:08 [INFO] src.models.retriever.Harrier: Loading model: microsoft/harrier-oss-v1-0.6b on cuda:0
2026-06-03 23:18:08 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: microsoft/harrier-oss-v1-0.6b
2026-06

[GPU] 嵌入模型已释放
✅ BFP-Proxy 向量库构建完成
BFP-Proxy 向量库 Done


### Section 2.2 — BFP-APR 向量库

配置文件：`configs/database/bfp/apr.yml`  
语料库和模型文件需要在配置文件中进行设置

In [6]:
# ========== Section 2.2: 构建 BFP-APR 向量库 ==========
# 模型和 URI 均从 apr.yml 的 database.model / database.uri 读取，无需手动覆写

apr_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/apr.yml"))

# 路径绝对化
if not os.path.isabs(apr_db_config["database"]["uri"]):
    apr_db_config["database"]["uri"] = str(PROJECT_ROOT / apr_db_config["database"]["uri"])
if not os.path.isabs(apr_db_config["data"]["corpus"]):
    apr_db_config["data"]["corpus"] = str(PROJECT_ROOT / apr_db_config["data"]["corpus"])

apr_milvus = BFPMilvusClient(apr_db_config)
col_name = apr_db_config["database"]["collection_name"]

already_built = (
    apr_milvus.client.has_collection(col_name)
    and apr_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = apr_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] BFP-APR 向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(apr_db_config["data"]["corpus"])
    print(f"构建 BFP-APR 向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(apr_db_config)
    apr_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ BFP-APR 向量库构建完成")

apr_milvus.close()
print("BFP-APR 向量库 Done")

2026-06-04 00:06:02 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/gte-modernbert-base.yml 读取 dim=768
2026-06-04 00:06:02 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/apr/gte-modernbert-base.db
2026-06-04 00:06:03 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-06-04 00:06:13 [INFO] src.rag.bfp.milvus_client: Milvus client closed


[SKIP] BFP-APR 向量库已存在，共 51655 条
BFP-APR 向量库 Done


### Section 2.3 — Vul-Vinj 向量库

配置文件：`configs/database/vul/vinj.yml`  
语料库和模型文件需要在配置文件中进行设置

In [7]:
# ========== Section 2.3: 构建 Vul-Vinj 向量库 ==========

vinj_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/vul/vinj.yml"))

# 路径绝对化
if not os.path.isabs(vinj_db_config["database"]["uri"]):
    vinj_db_config["database"]["uri"] = str(PROJECT_ROOT / vinj_db_config["database"]["uri"])
if not os.path.isabs(vinj_db_config["data"]["corpus"]):
    vinj_db_config["data"]["corpus"] = str(PROJECT_ROOT / vinj_db_config["data"]["corpus"])

vinj_milvus = VulMilvusClient(vinj_db_config)
col_name = vinj_db_config["database"]["collection_name"]

already_built = (
    vinj_milvus.client.has_collection(col_name)
    and vinj_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = vinj_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] Vul-Vinj 向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(vinj_db_config["data"]["corpus"])
    print(f"构建 Vul-Vinj 向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(vinj_db_config)
    vinj_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ Vul-Vinj 向量库构建完成")

vinj_milvus.close()
print("Vul-Vinj 向量库 Done")

2026-06-04 00:06:13 [INFO] src.rag.vul.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-06-04 00:06:13 [INFO] src.rag.vul.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/vul/harrier-oss-v1-0.6b.db
2026-06-04 00:06:13 [INFO] src.rag.vul.milvus_client: Milvus client connected
2026-06-04 00:06:17 [INFO] src.rag.vul.milvus_client: Milvus client closed


[SKIP] Vul-Vinj 向量库已存在，共 17576 条
Vul-Vinj 向量库 Done


## Section 3 — 代理查询集 & 测试查询集构建

### Section 3.1 — BFP 查询集 ↔ CVE 实例匹配

以 Vul 语料库中每条 CVE 实例的 `vul_code` 作为查询，从 BFP-Proxy 向量库检索相关 BFP，并将结果保存为检索结果文件。

In [8]:
# ========== Section 3.1: BFP 查询集 ↔ CVE 实例匹配 ==========
# 使用 proxy.yml 配置（data.corpus=BFP索引库, data.query=CVE实例, query_target=vul_code）

proxy_db_config_s3 = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/proxy.yml"))

# 路径绝对化
if not os.path.isabs(proxy_db_config_s3["database"]["uri"]):
    proxy_db_config_s3["database"]["uri"] = str(PROJECT_ROOT / proxy_db_config_s3["database"]["uri"])
if not os.path.isabs(proxy_db_config_s3["data"]["corpus"]):
    proxy_db_config_s3["data"]["corpus"] = str(PROJECT_ROOT / proxy_db_config_s3["data"]["corpus"])
if not os.path.isabs(proxy_db_config_s3["data"]["query"]):
    proxy_db_config_s3["data"]["query"] = str(PROJECT_ROOT / proxy_db_config_s3["data"]["query"])

# 推导输出文件路径
top_k_s3              = proxy_db_config_s3["retrieval"].get("top_k", 10)
query_target_s3       = proxy_db_config_s3["retrieval"].get("query_target", "vul_code")
retrieval_strategy_s3 = proxy_db_config_s3["retrieval"].get("retrieval_strategy", "dense")
alpha_str_s3          = "1" if retrieval_strategy_s3 == "dense" else (
                        "0" if retrieval_strategy_s3 == "sparse" else
                        str(proxy_db_config_s3["retrieval"].get("alpha_weight", 0.5)))

# 加载查询数据（CVE 实例）
query_data_s3 = load_jsonl(proxy_db_config_s3["data"]["query"])
print(f"CVE 实例查询集: {len(query_data_s3)} 条")

# 加载 embedder 以获取模型名（用于构造结果文件名）
proxy_embedder_s3  = load_embedder_from_db_config(proxy_db_config_s3)
model_short_name   = proxy_embedder_s3.get_model_name(with_provider=False)

retrieval_results_filename = (
    f"num_{len(query_data_s3)}_alpha_{alpha_str_s3}"
    f"_target_{query_target_s3}_retriever_{model_short_name}_retrieval_results.json"
)
# PROXY_RETRIEVAL_RESULTS_PATH = PROJECT_ROOT / "data/query/white" / retrieval_results_filename
PROXY_RETRIEVAL_RESULTS_PATH = PROJECT_ROOT / "data/query/gray" / retrieval_results_filename

print(f"检索结果将保存至: {PROXY_RETRIEVAL_RESULTS_PATH}")

if PROXY_RETRIEVAL_RESULTS_PATH.exists():
    release_embedder(proxy_embedder_s3)
    print(f"[SKIP] 检索结果文件已存在")
    proxy_retrieval_results = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))
    print(f"       加载 {len(proxy_retrieval_results)} 条")
else:
    # 打开 BFP-Proxy 向量库并执行检索
    proxy_milvus_s3 = BFPMilvusClient(proxy_db_config_s3)
    retriever_s3    = BFPRetriever(proxy_embedder_s3, proxy_milvus_s3)

    print(f"执行密集检索: top_k={top_k_s3}, query_target={query_target_s3}")
    proxy_retrieval_results = retriever_s3.dense_retrieve(
        query_data_s3, top_k=top_k_s3, query_target=query_target_s3
    )

    PROXY_RETRIEVAL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_json(proxy_retrieval_results, str(PROXY_RETRIEVAL_RESULTS_PATH))
    print(f"✅ 检索结果已保存: {len(proxy_retrieval_results)} 条 → {PROXY_RETRIEVAL_RESULTS_PATH}")

    proxy_milvus_s3.close()
    release_embedder(proxy_embedder_s3)

print("Section 3.1 Done")

2026-06-04 00:06:32 [INFO] src.utils.io: Loaded 17576 records from <PROJECT_ROOT>/data/corpus/vul/clustered_rag_data.jsonl
2026-06-04 00:06:32 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-06-04 00:06:32 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-06-04 00:06:32 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-06-04 00:06:32 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-06-04 00:06:32 [INFO] src.utils.device: 预筛选后的设备池: ['cuda:0']
2026-06-04 00:06:32 [INFO] src.utils.device: 设备池探测成功: cuda:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8187 MiB)
2026-06-04 00:06:32 [INFO] src.models.retriever.Harrier: Loading model: microsoft/harrier-oss-v1-0.6b on cuda:0
2026-06-04 00:06:32 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: microsoft/harrier-oss-v1-0.6b


CVE 实例查询集: 17576 条


2026-06-04 00:06:45 [INFO] sentence_transformers.SentenceTransformer: 3 prompts are loaded, with the keys: ['web_search_query', 'sts_query', 'bitext_query']
2026-06-04 00:06:45 [INFO] src.models.retriever.Harrier: Model loaded. dim=1024, task=code2code
2026-06-04 00:06:45 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-06-04 00:06:45 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/proxy/gray/harrier-oss-v1-0.6b.db
2026-06-04 00:06:45 [INFO] src.rag.bfp.milvus_client: Milvus client connected


检索结果将保存至: <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json
执行密集检索: top_k=10, query_target=vul_code


APR dense batch retrieval: 100%|██████████| 69/69 [25:16<00:00, 21.99s/it]
2026-06-04 00:32:02 [INFO] src.rag.bfp.retriever: Dense batch retrieval completed: 17576 queries, top_k=10, shards=69
2026-06-04 00:32:54 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json
2026-06-04 00:32:54 [INFO] src.rag.bfp.milvus_client: Milvus client closed


✅ 检索结果已保存: 17576 条 → <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json
[GPU] 嵌入模型已释放
Section 3.1 Done


### Section 3.2 v3 — CWE-MIXED 汇总去重 + 8:2 划分

遍历 BFP↔CVE 检索结果，**跨 CWE_LIST 中所有 CWE 类型汇总命中的 BFP**，过滤低分条目并去重，统一存入单键 `CWE-MIXED`，最后按 8:2 随机划分为代理查询集和测试查询集。（取自 v2 的单桶逻辑）

In [9]:
# ========== Section 3.2 v3: CWE-MIXED 汇总去重 + CVE 关联数过滤 + 8:2 分割 ==========
# 注意幂等：若 proxy/test 文件已存在则直接跳过，新阈值生效需先删除两个文件后重跑：
#   data/query/gray/proxy/proxy_query_set_v3.json
#   data/query/gray/test/test_query_set_v3.json

CWE_LIST                 = ["CWE-125", "CWE-20", "CWE-119", "CWE-787", "CWE-416"]
RELEVANCE_THRESHOLD      = 0.80
TOP_K_FILTER             = 10
# 每条 BFP 需被至少 CVE_ASSOCIATION_THRESHOLD 个不同的合格 CVE 实例召回才纳入候选
CVE_ASSOCIATION_THRESHOLD = 3
SPLIT_RATIO              = 0.8
RANDOM_SEED              = 42

# PROXY_QUERY_SET_PATH_V3 = PROJECT_ROOT / "data/query/white/proxy/proxy_query_set_v3.json"
PROXY_QUERY_SET_PATH_V3 = PROJECT_ROOT / "data/query/gray/proxy/proxy_query_set_v3.json"

# TEST_QUERY_SET_PATH_V3  = PROJECT_ROOT / "data/query/white/test/test_query_set_v3.json"
TEST_QUERY_SET_PATH_V3  = PROJECT_ROOT / "data/query/gray/test/test_query_set_v3.json"


def collect_bfps_cwe_mixed(
    results: list,
    target_cwes: list,
    threshold: float,
    top_k: int,
    assoc_threshold: int = 1,
) -> list:
    """从 vul 检索结果中收集 BFP（hit 侧）。

    只要该 BFP 被 target_cwes 中任意一种 CVE 实例以 score ≥ threshold 且 rank ≤ top_k
    召回，即纳入候选；按 doc_id 去重并保留最高分。
    仅保留被至少 assoc_threshold 个不同 CVE 实例（以 entity.id 去重）召回的 BFP。
    """
    bfp_best  = {}                 # doc_id -> 最高分 hit
    bfp_assoc = defaultdict(set)   # doc_id -> {cve_instance_id}
    target_set = set(target_cwes)
    for entry in results:
        if entry.get("entity", {}).get("cwe_id") not in target_set:
            continue
        cve_inst_id = entry.get("entity", {}).get("id")
        for rank, hit in enumerate(entry.get("retrieval_results", []), start=1):
            if rank > top_k:
                break
            if hit["score"] < threshold:
                continue
            doc_id = hit["doc_id"]
            bfp_assoc[doc_id].add(cve_inst_id)
            if doc_id not in bfp_best or hit["score"] > bfp_best[doc_id]["score"]:
                bfp_best[doc_id] = hit
    return [
        {
            "id"        : h["doc_id"],
            "source"    : h["source"],
            "buggy_code": h["buggy_code"],
            "fixed_code": h["fixed_code"],
        }
        for did, h in bfp_best.items()
        if len(bfp_assoc[did]) >= assoc_threshold
    ]


def split_proxy_test(bfps: list, ratio: float, seed: int) -> tuple:
    """固定种子随机打乱后按比例分割为 proxy 和 test。"""
    shuffled = bfps[:]
    random.Random(seed).shuffle(shuffled)
    cut = int(len(shuffled) * ratio)
    return shuffled[:cut], shuffled[cut:]


if PROXY_QUERY_SET_PATH_V3.exists() and TEST_QUERY_SET_PATH_V3.exists():
    print(f"[SKIP] proxy/test 查询集 v3 已存在")
    with open(PROXY_QUERY_SET_PATH_V3, encoding="utf-8") as f:
        proxy_query_set_v3 = json.load(f)
    with open(TEST_QUERY_SET_PATH_V3, encoding="utf-8") as f:
        test_query_set_v3 = json.load(f)
    print(f"  proxy CWE-MIXED: {len(proxy_query_set_v3.get('CWE-MIXED', []))} 条")
    print(f"  test  CWE-MIXED: {len(test_query_set_v3.get('CWE-MIXED', []))} 条")
else:
    # 加载检索结果
    results_s32 = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))

    bfps = collect_bfps_cwe_mixed(
        results_s32, CWE_LIST, RELEVANCE_THRESHOLD, TOP_K_FILTER,
        assoc_threshold=CVE_ASSOCIATION_THRESHOLD,
    )
    proxy, test = split_proxy_test(bfps, SPLIT_RATIO, RANDOM_SEED)

    proxy_query_set_v3 = {"CWE-MIXED": proxy}
    test_query_set_v3  = {"CWE-MIXED": test}

    PROXY_QUERY_SET_PATH_V3.parent.mkdir(parents=True, exist_ok=True)
    TEST_QUERY_SET_PATH_V3.parent.mkdir(parents=True, exist_ok=True)

    with open(PROXY_QUERY_SET_PATH_V3, "w", encoding="utf-8") as f:
        json.dump(proxy_query_set_v3, f, ensure_ascii=False, indent=2)
    with open(TEST_QUERY_SET_PATH_V3, "w", encoding="utf-8") as f:
        json.dump(test_query_set_v3, f, ensure_ascii=False, indent=2)

    print(f"\n===== CWE-MIXED proxy / test 查询集 v3 导出汇总 =====")
    print(f"  过滤: score >= {RELEVANCE_THRESHOLD}，rank ∈ [1, {TOP_K_FILTER}]")
    print(f"  关联合格 CVE 实例数 >= {CVE_ASSOCIATION_THRESHOLD}")
    print(f"  目标 CWE: {CWE_LIST}")
    print(f"  分割: proxy {SPLIT_RATIO*100:.0f}% / test {100-SPLIT_RATIO*100:.0f}% (seed={RANDOM_SEED})")
    print(f"  {'-'*42}")
    print(f"  CWE-MIXED  total={len(bfps):>6}  proxy={len(proxy):>6}  test={len(test):>6}")
    print(f"  proxy → {PROXY_QUERY_SET_PATH_V3}")
    print(f"  test  → {TEST_QUERY_SET_PATH_V3}")

print("Section 3.2 v3 Done")

2026-06-04 00:41:34 [INFO] src.utils.io: Stream-loaded 17576 records from <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json



===== CWE-MIXED proxy / test 查询集 v3 导出汇总 =====
  过滤: score >= 0.8，rank ∈ [1, 10]
  关联合格 CVE 实例数 >= 3
  目标 CWE: ['CWE-125', 'CWE-20', 'CWE-119', 'CWE-787', 'CWE-416']
  分割: proxy 80% / test 20% (seed=42)
  ------------------------------------------
  CWE-MIXED  total=   815  proxy=   652  test=   163
  proxy → <PROJECT_ROOT>/data/query/gray/proxy/proxy_query_set_v3.json
  test  → <PROJECT_ROOT>/data/query/gray/test/test_query_set_v3.json
Section 3.2 v3 Done


### Section 3.3 — BFP ↔ 合格 CVE 实例关联数统计

对 Section 3.2 v3 产出的「代理查询集 + 测试查询集」（`CWE-MIXED` 桶），按照与 3.2 完全一致的过滤口径（`score >= RELEVANCE_THRESHOLD`、`rank <= TOP_K_FILTER`、`cwe_id ∈ CWE_LIST`）重新计算每条 BFP 关联的**不同合格 CVE 实例数**，并输出最小/最大/中位数/平均等描述性统计量。

> 本 cell 自包含，可独立重跑（不依赖 3.2 中间变量）。

In [11]:
# ========== Section 3.3: BFP ↔ 合格 CVE 实例关联数统计 ==========
# 自包含：从检索结果文件重新构建 doc_id -> {cve_instance_id} 关联映射，
# 然后对 proxy+test 查询集内的每条 BFP 统计其关联的合格 CVE 实例数。

import statistics as _stats

# ── 1. 重新加载 BFP↔CVE 检索结果（避免依赖 3.2 skip 路径下可能缺失的中间变量）──
_results_s33 = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))

# ── 2. 按与 3.2 完全一致的口径构建 doc_id -> {cve_instance_id} 关联映射 ──
_target_set_s33 = set(CWE_LIST)
_assoc_s33 = defaultdict(set)  # doc_id -> {cve_instance_id}
for _entry in _results_s33:
    if _entry.get("entity", {}).get("cwe_id") not in _target_set_s33:
        continue
    _cve_inst_id = _entry.get("entity", {}).get("id")
    for _rank, _hit in enumerate(_entry.get("retrieval_results", []), start=1):
        if _rank > TOP_K_FILTER:
            break
        if _hit["score"] < RELEVANCE_THRESHOLD:
            continue
        _assoc_s33[_hit["doc_id"]].add(_cve_inst_id)

# ── 3. 加载 proxy + test 查询集（兼容 skip 路径：优先用内存变量，否则从磁盘加载）──
try:
    _proxy_bfps = proxy_query_set_v3.get("CWE-MIXED", [])
    _test_bfps  = test_query_set_v3.get("CWE-MIXED", [])
except NameError:
    with open(PROXY_QUERY_SET_PATH_V3, encoding="utf-8") as _f:
        _proxy_bfps = json.load(_f).get("CWE-MIXED", [])
    with open(TEST_QUERY_SET_PATH_V3, encoding="utf-8") as _f:
        _test_bfps = json.load(_f).get("CWE-MIXED", [])

_all_bfp_ids = [b["id"] for b in _proxy_bfps] + [b["id"] for b in _test_bfps]

# ── 4. 统计每条 BFP 的关联合格 CVE 实例数 ──
_counts_s33 = [len(_assoc_s33.get(bid, set())) for bid in _all_bfp_ids]

# ── 5. 打印描述性统计 ──
CVE_ASSOC_COUNT_THRESHOLD = 3  # 关联 CVE 实例数阈值 n

if not _counts_s33:
    print("[WARN] 未找到任何 BFP，请先运行 Section 3.1 和 Section 3.2 v3")
else:
    _above_n = sum(1 for c in _counts_s33 if c > CVE_ASSOC_COUNT_THRESHOLD)
    print(f"===== BFP ↔ 合格 CVE 实例关联数统计（proxy={len(_proxy_bfps)} + test={len(_test_bfps)} = {len(_all_bfp_ids)} 条）=====")
    print(f"  过滤口径: score >= {RELEVANCE_THRESHOLD}，rank ∈ [1, {TOP_K_FILTER}]，cwe_id ∈ {CWE_LIST}")
    print(f"  {'-'*56}")
    print(f"  最小值 (min)   : {min(_counts_s33)}")
    print(f"  最大值 (max)   : {max(_counts_s33)}")
    print(f"  中位数 (median): {_stats.median(_counts_s33):.1f}")
    print(f"  平均值 (mean)  : {_stats.mean(_counts_s33):.2f}")
    print(f"  关联 CVE 数 > {CVE_ASSOC_COUNT_THRESHOLD} : {_above_n} 个（占 {_above_n / len(_counts_s33) * 100:.1f}%）")

2026-06-04 00:48:54 [INFO] src.utils.io: Stream-loaded 17576 records from <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json


===== BFP ↔ 合格 CVE 实例关联数统计（proxy=652 + test=163 = 815 条）=====
  过滤口径: score >= 0.8，rank ∈ [1, 10]，cwe_id ∈ ['CWE-125', 'CWE-20', 'CWE-119', 'CWE-787', 'CWE-416']
  --------------------------------------------------------
  最小值 (min)   : 3
  最大值 (max)   : 64
  中位数 (median): 5.0
  平均值 (mean)  : 7.06
  关联 CVE 数 > 3 : 617 个（占 75.7%）


## Section 4 v3 — 投毒目标 & 正样本选择（CWE-MIXED 单桶 + 反向索引 τ 阈值）

对 Section 3.2 v3 产出的单一 `CWE-MIXED` 代理查询集：以每条 BFP 的 `buggy_code` 检索 BFP-APR 库，构建 `doc_id → 关联代理查询` 反向索引，保留被关联数 **严格大于** `ASSOCIATION_THRESHOLD`（默认 2）的 BFP 作为投毒目标，并把关联到的代理查询去重后作为 `retrieval_attack.positive_samples`。（取自 v4 的反向索引方法，但遍历对象由逐 CWE 改为单一 `CWE-MIXED`）

In [23]:
# ========== Section 4 v3: 投毒目标 + 正样本选择（CWE-MIXED 单桶 + 反向索引 τ 阈值） ==========
# 产出：data/query/gray/target/poison_targets_set_v3.json
# 模型和 URI 均从 data_prep_v3/target.yml 的 database.model / database.uri 读取

target_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/data_prep_v3/target.yml"))

# 路径绝对化
if not os.path.isabs(target_db_config["database"]["uri"]):
    target_db_config["database"]["uri"] = str(PROJECT_ROOT / target_db_config["database"]["uri"])
if not os.path.isabs(target_db_config["data"]["corpus"]):
    target_db_config["data"]["corpus"] = str(PROJECT_ROOT / target_db_config["data"]["corpus"])
if not os.path.isabs(target_db_config["data"]["query"]):
    target_db_config["data"]["query"] = str(PROJECT_ROOT / target_db_config["data"]["query"])

MIXED_KEY                 = "CWE-MIXED"
# ASSOCIATION_THRESHOLD     = 2
ASSOCIATION_THRESHOLD     = 3
top_k_target              = target_db_config["retrieval"].get("top_k", 15)
retrieval_strategy_target = target_db_config["retrieval"].get("retrieval_strategy", "dense")
query_target_target       = target_db_config["retrieval"].get("query_target", "buggy_code")

# POISON_TARGETS_OUT_V3 = PROJECT_ROOT / "data/query/white/target/poison_targets_set_v3.json"
POISON_TARGETS_OUT_V3 = PROJECT_ROOT / "data/query/gray/target/poison_targets_set_v3.json"

if POISON_TARGETS_OUT_V3.exists():
    print(f"[SKIP] 投毒目标文件 v3 已存在：{POISON_TARGETS_OUT_V3}")
    poison_targets_v3 = load_json(str(POISON_TARGETS_OUT_V3))
    print(f"  {MIXED_KEY}: {len(poison_targets_v3.get(MIXED_KEY, []))} 条")
else:
    # 加载 v3 代理查询集（单桶 CWE-MIXED）
    proxy_data_s4 = load_json(str(PROXY_QUERY_SET_PATH_V3))
    proxy_queries = proxy_data_s4.get(MIXED_KEY, [])
    if not proxy_queries:
        raise ValueError(f"{MIXED_KEY} 子集为空，无法进行投毒目标选择")

    # 打开 BFP-APR 向量库
    target_milvus    = BFPMilvusClient(target_db_config)
    target_embedder  = load_embedder_from_db_config(target_db_config)
    target_retriever = BFPRetriever(target_embedder, target_milvus)

    print(f"\n{'='*60}")
    print(f"[{MIXED_KEY}] 开始处理，代理查询数: {len(proxy_queries)}")

    # ── 步骤1: 以 buggy_code 检索 BFP 知识库 ──
    if retrieval_strategy_target == "dense":
        retrieval_results_t = target_retriever.dense_retrieve(
            proxy_queries, top_k=top_k_target, query_target=query_target_target
        )
    elif retrieval_strategy_target == "sparse":
        retrieval_results_t = target_retriever.sparse_retrieve(
            proxy_queries, top_k=top_k_target, query_target=query_target_target
        )
    else:
        alpha_t = target_db_config["retrieval"].get("alpha_weight", 0.85)
        retrieval_results_t = target_retriever.hybrid_retrieve(
            proxy_queries, top_k=top_k_target, alpha=alpha_t,
            query_target=query_target_target
        )

    # ── 步骤2: 反向索引 ──
    reverse_index = defaultdict(list)
    for result in retrieval_results_t:
        proxy_entity = result["entity"]
        for hit in result["retrieval_results"]:
            reverse_index[hit["doc_id"]].append({"proxy": proxy_entity, "hit": hit})

    print(f"[{MIXED_KEY}] 反向索引构建完成，涉及候选 BFP: {len(reverse_index)} 个")

    # ── 步骤3: 关联数量阈值过滤 ──
    targets = []
    for doc_id, associations in reverse_index.items():
        if len(associations) <= ASSOCIATION_THRESHOLD:
            continue

        best = max(associations, key=lambda x: x["hit"]["score"])
        best_hit = best["hit"]
        target_entity = {
            "id"        : best_hit["doc_id"],
            "source"    : best_hit["source"],
            "buggy_code": best_hit["buggy_code"],
            "fixed_code": best_hit["fixed_code"],
        }

        seen_proxy_ids = set()
        positive_samples = []
        for a in associations:
            pid = a["proxy"]["id"]
            if pid not in seen_proxy_ids:
                seen_proxy_ids.add(pid)
                positive_samples.append({
                    "id"        : a["proxy"]["id"],
                    "source"    : a["proxy"]["source"],
                    "buggy_code": a["proxy"]["buggy_code"],
                    "fixed_code": a["proxy"]["fixed_code"],
                })

        targets.append({
            "entity": target_entity,
            "retrieval_attack": {"positive_samples": positive_samples},
        })

    poison_targets_v3 = {MIXED_KEY: targets}
    print(f"[{MIXED_KEY}] 投毒目标筛选完成: {len(targets)} 个（关联数 > {ASSOCIATION_THRESHOLD}）")

    target_milvus.close()
    release_embedder(target_embedder)

    # ── 写出 ──
    POISON_TARGETS_OUT_V3.parent.mkdir(parents=True, exist_ok=True)
    save_json(poison_targets_v3, str(POISON_TARGETS_OUT_V3))
    print(f"\n✅ 投毒目标文件 v3 已保存: {POISON_TARGETS_OUT_V3}")

    print(f"\n===== 汇总 =====")
    print(f"  {'CWE':<12} {'代理查询数':>10} {'入选目标数':>10}")
    print(f"  {'-'*36}")
    print(f"  {MIXED_KEY:<12} {len(proxy_queries):>10} {len(targets):>10}")

print("\nSection 4 v3 Done — 数据准备管线全部完成 ✅")

2026-06-04 01:14:30 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/gray/proxy/proxy_query_set_v3.json
2026-06-04 01:14:30 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/gte-modernbert-base.yml 读取 dim=768
2026-06-04 01:14:30 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/apr/gte-modernbert-base.db
2026-06-04 01:14:30 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-06-04 01:14:30 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-06-04 01:14:30 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-06-04 01:14:30 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-06-04 01:14:30 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-06-04 01:14:30 [INFO] src.utils.device: 预筛选后的设备池: ['cuda:0']
2026-06-04 01:14:30 [INFO] src.utils.device: 设备池探测成功: cuda:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8187 MiB)
2026-06-04 01:14:30 [INFO] src.models.retrie


[CWE-MIXED] 开始处理，代理查询数: 652


APR dense batch retrieval: 100%|██████████| 3/3 [00:42<00:00, 14.32s/it]
2026-06-04 01:15:24 [INFO] src.rag.bfp.retriever: Dense batch retrieval completed: 652 queries, top_k=15, shards=3
2026-06-04 01:15:24 [INFO] src.rag.bfp.milvus_client: Milvus client closed


[CWE-MIXED] 反向索引构建完成，涉及候选 BFP: 4755 个
[CWE-MIXED] 投毒目标筛选完成: 634 个（关联数 > 3）
[GPU] 嵌入模型已释放


2026-06-04 01:15:26 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v3.json



✅ 投毒目标文件 v3 已保存: <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v3.json

===== 汇总 =====
  CWE               代理查询数      入选目标数
  ------------------------------------
  CWE-MIXED           652        634

Section 4 v3 Done — 数据准备管线全部完成 ✅


In [27]:
# ========== 正样本数量统计分析 ==========
# 统计 poison_targets_set_v3.json 中每个投毒目标所拥有的正样本数量分布

import statistics

_targets = load_json(str(POISON_TARGETS_OUT_V3))
_pos_counts = [
    len(t.get("retrieval_attack", {}).get("positive_samples", []))
    for t in _targets.get(MIXED_KEY, [])
]

POSITIVE_SAMPLE_THRESHOLD = 5  # 正样本数量阈值 n

if not _pos_counts:
    print("[WARN] 未找到任何投毒目标，请先运行 Section 4 v3")
else:
    _above_n = sum(1 for c in _pos_counts if c > POSITIVE_SAMPLE_THRESHOLD)
    print(f"===== 正样本数量统计（{MIXED_KEY}，共 {len(_pos_counts)} 个投毒目标）=====")
    print(f"  最小值 (min)   : {min(_pos_counts)}")
    print(f"  最大值 (max)   : {max(_pos_counts)}")
    print(f"  中位数 (median): {statistics.median(_pos_counts):.1f}")
    print(f"  平均值 (mean)  : {statistics.mean(_pos_counts):.2f}")
    print(f"  正样本数 > {POSITIVE_SAMPLE_THRESHOLD} : {_above_n} 个（占 {_above_n / len(_pos_counts) * 100:.1f}%）")

2026-06-04 01:18:44 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v3.json


===== 正样本数量统计（CWE-MIXED，共 634 个投毒目标）=====
  最小值 (min)   : 4
  最大值 (max)   : 24
  中位数 (median): 5.0
  平均值 (mean)  : 6.00
  正样本数 > 5 : 279 个（占 44.0%）
